# More fun with profiles

Here's some code. See if you can figure out what's happening in the profile! Make sure to run this on a TPU v2-8 Colab runtime!

*Try running this code without looking at it first and look at the profile.*

In [1]:
import jax
import jax.numpy as jnp
import jax.sharding as shd

P = shd.PartitionSpec

mesh = jax.make_mesh((2, 4), ("x", "y"))

d_model = 8192
batch = 1
ffw_mult = 4

cpu_device = jax.devices("cpu")[0]

cpu_x = jnp.zeros((batch, d_model), dtype=jnp.bfloat16, device=cpu_device)
cpu_w1 = jnp.zeros((ffw_mult * d_model, d_model), dtype=jnp.bfloat16, device=cpu_device)
cpu_w2 = jnp.zeros((d_model, ffw_mult * d_model), dtype=jnp.bfloat16, device=cpu_device)


def matmul(w1, w2, x):
    return jnp.einsum("wf,bf->bw", w2, jnp.einsum("fw,bw->bf", w1, x))


def make_sharding():
    model_axes = ("x", "y")
    return (
        shd.NamedSharding(mesh, P(model_axes, None)),
        shd.NamedSharding(mesh, P(None, model_axes)),
        shd.NamedSharding(
            mesh,
            P(
                None,
            ),
        ),
    )


w1_sharding, w2_sharding, x_sharding = make_sharding()
x, w1, w2 = (
    jax.device_put(cpu_x, x_sharding),
    jax.device_put(cpu_w1, w1_sharding),
    jax.device_put(cpu_w2, w2_sharding),
)

compiled = jax.jit(matmul, in_shardings=make_sharding()).lower(w1, w2, x).compile()
result = compiled(w1, w2, x)

In [2]:
with jax.profiler.trace("/tmp/tensorboard"):
    y = compiled(w1, w2, x)
    y.block_until_ready()

2026-07-11 06:14:48.754988: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-11 06:14:48.908853: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [3]:
%load_ext tensorboard

In [5]:
%tensorboard --logdir=/tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 11234), started 0:00:12 ago. (Use '!kill 11234' to kill it.)